# Historical PN/GPN/AMGPN/matching comparison
Isolated from `GPN_V4_test.ipynb`, original evaluation cell 3. Configuration keys retain their original identifiers. This notebook contains one experiment axis; historical evaluator contracts still need execution validation.


In [ ]:
# 混淆矩阵绘制为多任务结果，当前为单任务


'\n多模型串行评估脚本 (精简版)\n===========================\n直接按照示例格式配置和运行\n\n使用方法:\n<configured>. 修改 DATASET_PATH 和 MODEL_DIR\n<configured>. 配置 model_paths, trainer_configs, model_initializers\n<configured>. 运行脚本\n'
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
from amgpn.evaluation.compare import *
from amgpn.data.episodes import UAVDataset, MetaDataset
from amgpn.evaluation.analysis_trainer import GPNTrainer, GPN_Optimized
from amgpn.comparisons.matching import MN_Backbone,MatchingNetworkTrainer
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
sys.path.append(_cfg_require('GPN_V4_test.ipynb.cell2.module.path'))  # 添加代码路径

# 数据集和模型路径 - 根据实际情况修改
# Historical configuration omitted; provide values through private configuration.
DATASET_PATH = _cfg_require('GPN_V4_test.ipynb.cell2.module.DATASET_PATH')
MODEL_DIR = _cfg_require('GPN_V4_test.ipynb.cell2.module.MODEL_DIR')

# 评估参数
N_WAY = _cfg_require('GPN_V4_test.ipynb.cell2.module.N_WAY')
K_SHOTS = _cfg_require('GPN_V4_test.ipynb.cell2.module.K_SHOTS')  # 要测试的K-shot
N_TRIALS = _cfg_require('GPN_V4_test.ipynb.cell2.module.N_TRIALS')
Q_QUERY = _cfg_require('GPN_V4_test.ipynb.cell2.module.Q_QUERY')
LR=_cfg_require('GPN_V4_test.ipynb.cell2.module.LR')
SEED = _cfg_require('GPN_V4_test.ipynb.cell2.module.SEED')


In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#第二版
#主要对比，<configured>基础模型main-result,折线图


# Historical configuration omitted; provide values through private configuration.
print("Loading dataset...")
test_dataset = UAVDataset(
    data_dir_path=DATASET_PATH,
    seed=SEED,
    is_train=False,
    train_ratio=_cfg_require('GPN_V4_test.ipynb.cell3.module.train_ratio'),
    max_sample_count=_cfg_require('GPN_V4_test.ipynb.cell3.module.max_sample_count')
)
print(f"Test dataset size: {len(test_dataset)}")

# Historical configuration omitted; provide values through private configuration.

# 模型路径列表
model_paths = [
    {'name': 'MN','path': _cfg_format('GPN_V4_test.ipynb.cell3.module.path_template', field0=MODEL_DIR)},
    {'name': 'AMGPN (τ=<configured>)', 'path': _cfg_format('GPN_V4_test.ipynb.cell3.module.path_template__2', field0=MODEL_DIR)},
    {'name': 'GPN', 'path': _cfg_format('GPN_V4_test.ipynb.cell3.module.path_template__3', field0=MODEL_DIR)},
    {'name': 'ProtoNet', 'path': _cfg_format('GPN_V4_test.ipynb.cell3.module.path_template__4', field0=MODEL_DIR)},
]

# Trainer配置列表（与model_paths一一对应）
trainer_configs = _cfg_require('GPN_V4_test.ipynb.cell3.module.trainer_configs')

# 模型初始化函数列表
model_initializers = [
    lambda: MN_Backbone(feature_dim=_cfg_require('GPN_V4_test.ipynb.cell3.module.feature_dim')),
    lambda: GPN_Optimized(),
    lambda: GPN_Optimized(),
    lambda: GPN_Optimized(),    
    # lambda: GPN_Optimized(),
    # lambda: GPN_Optimized(),
]

# Trainer类列表（如果不同模型用不同Trainer）
trainer_classes = [
    MatchingNetworkTrainer,
    GPNTrainer,  # AMGPN
    GPNTrainer,  # GPN
    GPNTrainer,  # ProtoNet
]

# Historical configuration omitted; provide values through private configuration.
results_df = test_multiple_models(
    test_dataset=test_dataset,
    model_paths=model_paths,
    trainer_configs=trainer_configs,
    model_initializers=model_initializers,
    trainer_classes=trainer_classes,
    n_way=N_WAY,
    k_shots=K_SHOTS,
    n_trials=N_TRIALS,
    q_query=Q_QUERY,
    device=_cfg_require('GPN_V4_test.ipynb.cell3.module.device'),
    seed=SEED,
    save_dir=_cfg_require('GPN_V4_test.ipynb.cell3.module.save_dir')
)

print("\nFinal Results DataFrame:")
print(results_df)